In [0]:
import os
from dotenv import load_dotenv
from functools import reduce
from pyspark.sql.functions import col


# Carregamento de variáveis de ambiente
load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# Dicionário de credenciais OAUTH para usar com Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Dicionário de DataSets que serão utilizados
tables = [
    "ecommerce_itens_pedido",
    "food_lotes_producao",
    "food_estoque_lojas",
    "physical_itens_venda_caixa"
]


In [0]:
# Gera os paths das tabelas
def generate_path(layer, table_name=None) -> str: 
    if layer == "silver":
        if table_name.startswith("ecommerce"):
            container = "squad2"
            folder = "grupo2/silver"
        elif table_name.startswith(("food", "physical")):
            container = "squad3"
            folder = "silver"
        else:
            raise ValueError(f"Tabela não reconhecida: {table_name}")
        path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/{folder}/{table_name}"
    elif layer == "ml":
        path = f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/ml/{table_name}"    
    else:
        raise ValueError(f"Camada inválida: {layer}")
    return path

# Leitura de datasets
def read_dataset(tables: list, layer="silver") -> dict:
    dataframes = {}
    for table_name in tables:
        df = (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(generate_path(layer, table_name=table_name))
        )
        dataframes[f"df_{table_name}_{layer}"] = df
    return dataframes

dataframes = read_dataset(tables)

df_food_estoque_lojas_silver = dataframes["df_food_estoque_lojas_silver"]
df_food_lotes_producao_silver = dataframes["df_food_lotes_producao_silver"]
df_ecommerce_itens_pedido_silver = dataframes["df_ecommerce_itens_pedido_silver"]
df_physical_itens_venda_caixa_silver = dataframes["df_physical_itens_venda_caixa_silver"]

In [0]:

# ============================================================
# NOTEBOOK 01 - EDA CONSOLIDATION
# Project: Category-Level Demand Forecasting
# Objective: Consolidate EDA results and validate Silver sources
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, DoubleType
)

# ------------------------------------------------------------
# 1. LOAD SILVER DATASETS
# ------------------------------------------------------------

datasets = {
    "ecommerce_itens_pedido": dataframes["df_ecommerce_itens_pedido_silver"],
    "physical_itens_venda_caixa": dataframes["df_physical_itens_venda_caixa_silver"],
    "food_estoque_lojas": dataframes["df_food_estoque_lojas_silver"],
    "food_lotes_producao": dataframes["df_food_food_lotes_producao_silver"]
}

# Candidate primary keys (validated only when present)
primary_keys = {
    "ecommerce_itens_pedido": ["id_item_pedido"],
    "physical_itens_venda_caixa": ["id_item_venda"],
    "food_estoque_lojas": [],
    "food_lotes_producao": ["id_lote"]
}

# Candidate business date columns
date_candidates = [
    "dt_pedido",
    "dt_venda",
    "dt_snapshot",
    "dt_fabricacao",
    "dt_validade"
]

numeric_types = [
    "int", "bigint", "long", "double",
    "float", "smallint", "decimal"
]

# ------------------------------------------------------------
# 2. OVERVIEW
# ------------------------------------------------------------

overview_results = []

for name, df in datasets.items():

    numeric_columns = [
        c for c, t in df.dtypes
        if any(t.startswith(nt) for nt in numeric_types)
    ]

    temporal_columns = [
        c for c, t in df.dtypes
        if "date" in t or "timestamp" in t
    ]

    overview_results.append((
        name,
        df.count(),
        len(df.columns),
        len(numeric_columns),
        len(temporal_columns)
    ))

df_overview = spark.createDataFrame(
    overview_results,
    [
        "dataset", "total_records", "total_columns",
        "numeric_columns", "temporal_columns"
    ]
)

print("1. DATASET OVERVIEW")
display(df_overview)


# ------------------------------------------------------------
# 3. NULL ANALYSIS
# ------------------------------------------------------------

null_results = []

for name, df in datasets.items():

    total_records = df.count()

    for column_name in df.columns:

        null_count = df.filter(
            F.col(column_name).isNull()
        ).count()

        null_percentage = (
            round(null_count / total_records * 100, 2)
            if total_records > 0 else 0.0
        )

        null_results.append((
            name,
            column_name,
            null_count,
            null_percentage
        ))

df_null_analysis = spark.createDataFrame(
    null_results,
    [
        "dataset", "column",
        "null_count", "null_percentage"
    ]
)

print("2. NULL ANALYSIS")
display(
    df_null_analysis
    .filter(F.col("null_count") > 0)
    .orderBy(F.desc("null_percentage"))
)


# ------------------------------------------------------------
# 4. DUPLICATE ANALYSIS
# ------------------------------------------------------------

duplicate_results = []

for name, df in datasets.items():

    total_records = df.count()
    distinct_records = df.distinct().count()

    duplicate_results.append((
        name,
        "complete_record",
        total_records - distinct_records
    ))

    for key_column in primary_keys[name]:

        if key_column in df.columns:

            duplicated_keys = (
                df.groupBy(key_column)
                .count()
                .filter(F.col("count") > 1)
                .count()
            )

            duplicate_results.append((
                name,
                key_column,
                duplicated_keys
            ))

df_duplicates = spark.createDataFrame(
    duplicate_results,
    ["dataset", "analysis_type", "duplicate_count"]
)

print("3. DUPLICATE ANALYSIS")
display(df_duplicates)


# ------------------------------------------------------------
# 5. TEMPORAL COVERAGE
# ------------------------------------------------------------

temporal_schema = StructType([
    StructField("dataset", StringType(), True),
    StructField("date_column", StringType(), True),
    StructField("min_date", StringType(), True),
    StructField("max_date", StringType(), True),
    StructField("distinct_dates", LongType(), True)
])

temporal_results = []

for name, df in datasets.items():

    available_dates = [
        c for c in date_candidates
        if c in df.columns
    ]

    if not available_dates:

        temporal_results.append((
            name, None, None, None, None
        ))

    for date_column in available_dates:

        result = df.select(
            F.min(F.to_date(F.col(date_column))).alias("min_date"),
            F.max(F.to_date(F.col(date_column))).alias("max_date"),
            F.countDistinct(
                F.to_date(F.col(date_column))
            ).alias("distinct_dates")
        ).first()

        temporal_results.append((
            name,
            date_column,
            str(result["min_date"]) if result["min_date"] else None,
            str(result["max_date"]) if result["max_date"] else None,
            result["distinct_dates"]
        ))

df_temporal_coverage = spark.createDataFrame(
    temporal_results,
    temporal_schema
)

print("4. TEMPORAL COVERAGE")
display(df_temporal_coverage)


# ------------------------------------------------------------
# 6. NUMERICAL DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

numeric_schema = StructType([
    StructField("dataset", StringType(), True),
    StructField("column", StringType(), True),
    StructField("valid_values", LongType(), True),
    StructField("minimum", DoubleType(), True),
    StructField("maximum", DoubleType(), True),
    StructField("average", DoubleType(), True),
    StructField("standard_deviation", DoubleType(), True)
])

numeric_results = []

for name, df in datasets.items():

    numeric_columns = [
        c for c, t in df.dtypes
        if any(t.startswith(nt) for nt in numeric_types)
    ]

    for column_name in numeric_columns:

        result = df.select(
            F.count(F.col(column_name)).alias("valid_values"),
            F.min(F.col(column_name)).alias("minimum"),
            F.max(F.col(column_name)).alias("maximum"),
            F.avg(F.col(column_name)).alias("average"),
            F.stddev(F.col(column_name)).alias("standard_deviation")
        ).first()

        numeric_results.append((
            name,
            column_name,
            result["valid_values"],
            float(result["minimum"]) if result["minimum"] is not None else None,
            float(result["maximum"]) if result["maximum"] is not None else None,
            float(result["average"]) if result["average"] is not None else None,
            float(result["standard_deviation"]) if result["standard_deviation"] is not None else None
        ))

df_numeric_analysis = spark.createDataFrame(
    numeric_results,
    numeric_schema
)

print("5. NUMERICAL DESCRIPTIVE STATISTICS")
display(df_numeric_analysis.orderBy("dataset", "column"))


# ------------------------------------------------------------
# 7. NUMERICAL QUALITY: NEGATIVE AND ZERO VALUES
# ------------------------------------------------------------

quality_schema = StructType([
    StructField("dataset", StringType(), True),
    StructField("column", StringType(), True),
    StructField("negative_count", LongType(), True),
    StructField("zero_count", LongType(), True)
])

quality_results = []

for name, df in datasets.items():

    numeric_columns = [
        c for c, t in df.dtypes
        if any(t.startswith(nt) for nt in numeric_types)
    ]

    for column_name in numeric_columns:

        result = df.agg(
            F.sum(
                F.when(F.col(column_name) < 0, 1).otherwise(0)
            ).alias("negative_count"),

            F.sum(
                F.when(F.col(column_name) == 0, 1).otherwise(0)
            ).alias("zero_count")
        ).first()

        quality_results.append((
            name,
            column_name,
            result["negative_count"] or 0,
            result["zero_count"] or 0
        ))

df_numeric_quality = spark.createDataFrame(
    quality_results,
    quality_schema
)

print("6. NUMERICAL QUALITY")
display(df_numeric_quality.orderBy("dataset", "column"))


# ------------------------------------------------------------
# 8. CONSOLIDATED SCHEMA
# ------------------------------------------------------------

schema_results = []

for name, df in datasets.items():

    for column_name, data_type in df.dtypes:

        schema_results.append((
            name,
            column_name,
            data_type
        ))

df_schema_consolidation = spark.createDataFrame(
    schema_results,
    ["dataset", "column", "data_type"]
)

print("7. CONSOLIDATED SCHEMA")
display(df_schema_consolidation.orderBy("dataset", "column"))


# ------------------------------------------------------------
# 9. SOURCE ASSESSMENT
# ------------------------------------------------------------

source_assessment = [
    (
        "ecommerce_itens_pedido",
        "Sales",
        "Primary source for e-commerce demand"
    ),
    (
        "physical_itens_venda_caixa",
        "Sales",
        "Primary source for physical store demand"
    ),
    (
        "food_estoque_lojas",
        "Inventory",
        "Stock availability and minimum stock analysis"
    ),
    (
        "food_lotes_producao",
        "Production",
        "Production and supply context"
    )
]

df_source_assessment = spark.createDataFrame(
    source_assessment,
    ["dataset", "business_role", "project_usage"]
)

print("8. SOURCE ASSESSMENT")
display(df_source_assessment)


# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("=" * 60)
print("EDA CONSOLIDATION COMPLETED")
print("=" * 60)

print(f"Datasets analyzed: {len(datasets)}")

for name, df in datasets.items():
    print(
        f"{name}: "
        f"{df.count()} records | "
        f"{len(df.columns)} columns"
    )

print("=" * 60)
print("NEXT STEP: SALES INTEGRATION")
print("=" * 60)


In [0]:

# ============================================================
# NOTEBOOK 02 - SALES INTEGRATION
# Project: Category-Level Demand Forecasting
# Objective: Prepare e-commerce and physical sales independently
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. LOAD SILVER DATASETS
# ------------------------------------------------------------

df_ecommerce = dataframes["df_ecommerce_itens_pedido_silver"]

df_physical_items = dataframes[
    "df_physical_itens_venda_caixa_silver"
]

# Search for the physical sales header DataFrame.
# It is needed only if the item table has no sale date.

header_candidates = [
    "df_physical_vendas_caixa_silver",
    "physical_vendas_caixa",
    "df_physical_vendas_caixa",
    "df_vendas_caixa_silver"
]

df_physical_header = None

for candidate in header_candidates:
    if candidate in dataframes:
        df_physical_header = dataframes[candidate]
        break


# ------------------------------------------------------------
# 2. VALIDATION FUNCTION
# ------------------------------------------------------------

def validate_columns(df, required_columns, dataset_name):

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing columns {missing_columns}. "
            f"Available columns: {df.columns}"
        )


# ------------------------------------------------------------
# 3. PREPARE E-COMMERCE SALES
# ------------------------------------------------------------

validate_columns(
    df_ecommerce,
    ["sku", "quantidade", "dt_pedido"],
    "E-commerce sales"
)

ecommerce_columns = df_ecommerce.columns

df_ecommerce_prepared = df_ecommerce.select(

    (
        F.col("id_item_pedido").cast("string")
        if "id_item_pedido" in ecommerce_columns
        else F.lit(None).cast("string")
    ).alias("sales_item_id"),

    F.col("sku").cast("string").alias("product_id"),

    F.to_date("dt_pedido").alias("sale_date"),

    F.col("quantidade").cast("double").alias("quantity_sold"),

    F.lit("ecommerce").alias("sales_channel"),

    (
        F.col("id_pedido").cast("string")
        if "id_pedido" in ecommerce_columns
        else F.lit(None).cast("string")
    ).alias("transaction_id"),

    (
        F.col("preco_unitario").cast("double")
        if "preco_unitario" in ecommerce_columns
        else F.lit(None).cast("double")
    ).alias("unit_price"),

    (
        F.col("receita_bruta").cast("double")
        if "receita_bruta" in ecommerce_columns
        else F.lit(None).cast("double")
    ).alias("gross_revenue"),

    (
        F.col("receita_liquida").cast("double")
        if "receita_liquida" in ecommerce_columns
        else F.lit(None).cast("double")
    ).alias("net_revenue")
)


# ------------------------------------------------------------
# 4. E-COMMERCE DATA QUALITY
# ------------------------------------------------------------

df_ecommerce_quality = (
    df_ecommerce_prepared

    .withColumn(
        "flag_missing_product",
        F.col("product_id").isNull()
        | (F.trim(F.col("product_id")) == "")
    )

    .withColumn(
        "flag_missing_date",
        F.col("sale_date").isNull()
    )

    .withColumn(
        "flag_missing_quantity",
        F.col("quantity_sold").isNull()
    )

    .withColumn(
        "flag_invalid_quantity",
        F.col("quantity_sold") <= 0
    )

    .withColumn(
        "flag_valid_record",
        ~F.col("flag_missing_product")
        & ~F.col("flag_missing_date")
        & ~F.col("flag_missing_quantity")
        & ~F.col("flag_invalid_quantity")
    )
)

df_ecommerce_valid = (
    df_ecommerce_quality
    .filter(F.col("flag_valid_record"))
    .drop(
        "flag_missing_product",
        "flag_missing_date",
        "flag_missing_quantity",
        "flag_invalid_quantity",
        "flag_valid_record"
    )
)

df_ecommerce_rejected = (
    df_ecommerce_quality
    .filter(~F.col("flag_valid_record"))
)


# ------------------------------------------------------------
# 5. PREPARE PHYSICAL SALES DATE
# ------------------------------------------------------------

validate_columns(
    df_physical_items,
    [
        "id_transacao",
        "codigo_barras_produto",
        "quantidade"
    ],
    "Physical sales items"
)

physical_columns = df_physical_items.columns

if "dt_venda" in physical_columns:

    df_physical_with_date = df_physical_items.withColumn(
        "sale_date",
        F.to_date("dt_venda")
    )

elif "sale_date" in physical_columns:

    df_physical_with_date = df_physical_items.withColumn(
        "sale_date",
        F.to_date("sale_date")
    )

else:

    if df_physical_header is None:

        raise ValueError(
            "Physical sales item table has no sale date. "
            "Load the physical sales header Silver DataFrame "
            "to obtain dt_venda using id_transacao."
        )

    validate_columns(
        df_physical_header,
        ["id_transacao", "dt_venda"],
        "Physical sales header"
    )

    # Ensure that each transaction has a single header record.
    duplicate_header_ids = (
        df_physical_header
        .groupBy("id_transacao")
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicate_header_ids > 0:
        raise ValueError(
            "Duplicate id_transacao values found in physical "
            "sales header. Resolve duplicates before joining."
        )

    df_header_dates = df_physical_header.select(
        F.col("id_transacao").cast("string")
        .alias("header_transaction_id"),

        F.to_date("dt_venda").alias("sale_date")
    )

    df_physical_with_date = (
        df_physical_items
        .join(
            df_header_dates,
            F.col("id_transacao").cast("string")
            == F.col("header_transaction_id"),
            "left"
        )
        .drop("header_transaction_id")
    )


# ------------------------------------------------------------
# 6. PREPARE PHYSICAL SALES
# ------------------------------------------------------------

physical_columns = df_physical_with_date.columns

df_physical_prepared = df_physical_with_date.select(

    (
        F.col("id_item_venda").cast("string")
        if "id_item_venda" in physical_columns
        else F.lit(None).cast("string")
    ).alias("sales_item_id"),

    F.col("codigo_barras_produto")
    .cast("string")
    .alias("product_id"),

    F.to_date("sale_date").alias("sale_date"),

    F.col("quantidade")
    .cast("double")
    .alias("quantity_sold"),

    F.lit("physical").alias("sales_channel"),

    F.col("id_transacao")
    .cast("string")
    .alias("transaction_id"),

    (
        F.col("preco_unitario_registro").cast("double")
        if "preco_unitario_registro" in physical_columns
        else F.lit(None).cast("double")
    ).alias("unit_price"),

    (
        F.col("valor_total_item").cast("double")
        if "valor_total_item" in physical_columns
        else F.lit(None).cast("double")
    ).alias("gross_revenue"),

    F.lit(None).cast("double").alias("net_revenue")
)


# ------------------------------------------------------------
# 7. PHYSICAL SALES DATA QUALITY
# ------------------------------------------------------------

df_physical_quality = (
    df_physical_prepared

    .withColumn(
        "flag_missing_product",
        F.col("product_id").isNull()
        | (F.trim(F.col("product_id")) == "")
    )

    .withColumn(
        "flag_missing_date",
        F.col("sale_date").isNull()
    )

    .withColumn(
        "flag_missing_quantity",
        F.col("quantity_sold").isNull()
    )

    .withColumn(
        "flag_invalid_quantity",
        F.col("quantity_sold") <= 0
    )

    .withColumn(
        "flag_valid_record",
        ~F.col("flag_missing_product")
        & ~F.col("flag_missing_date")
        & ~F.col("flag_missing_quantity")
        & ~F.col("flag_invalid_quantity")
    )
)

df_physical_valid = (
    df_physical_quality
    .filter(F.col("flag_valid_record"))
    .drop(
        "flag_missing_product",
        "flag_missing_date",
        "flag_missing_quantity",
        "flag_invalid_quantity",
        "flag_valid_record"
    )
)

df_physical_rejected = (
    df_physical_quality
    .filter(~F.col("flag_valid_record"))
)


# ------------------------------------------------------------
# 8. QUALITY SUMMARY FUNCTION
# ------------------------------------------------------------

def sales_quality_summary(df, channel_name):

    return df.agg(

        F.count("*").alias("total_records"),

        F.sum(
            F.when(F.col("flag_valid_record"), 1).otherwise(0)
        ).alias("valid_records"),

        F.sum(
            F.when(~F.col("flag_valid_record"), 1).otherwise(0)
        ).alias("rejected_records"),

        F.countDistinct("product_id").alias("distinct_products"),

        F.sum("quantity_sold").alias("total_quantity"),

        F.min("sale_date").alias("first_sale_date"),

        F.max("sale_date").alias("last_sale_date")

    ).withColumn(
        "sales_channel",
        F.lit(channel_name)
    )


df_ecommerce_summary = sales_quality_summary(
    df_ecommerce_quality,
    "ecommerce"
)

df_physical_summary = sales_quality_summary(
    df_physical_quality,
    "physical"
)


# ------------------------------------------------------------
# 9. DISPLAY E-COMMERCE RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("E-COMMERCE SALES")
print("=" * 60)

print("Prepared valid records")
display(df_ecommerce_valid)

print("Rejected records")
display(df_ecommerce_rejected)

print("E-commerce quality summary")
display(df_ecommerce_summary)


# ------------------------------------------------------------
# 10. DISPLAY PHYSICAL SALES RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("PHYSICAL SALES")
print("=" * 60)

print("Prepared valid records")
display(df_physical_valid)

print("Rejected records")
display(df_physical_rejected)

print("Physical sales quality summary")
display(df_physical_summary)


# ------------------------------------------------------------
# 11. FINAL VALIDATION
# ------------------------------------------------------------

print("=" * 60)
print("SALES PREPARATION COMPLETED")
print("=" * 60)

print(
    "E-commerce valid records:",
    df_ecommerce_valid.count()
)

print(
    "E-commerce rejected records:",
    df_ecommerce_rejected.count()
)

print(
    "Physical valid records:",
    df_physical_valid.count()
)

print(
    "Physical rejected records:",
    df_physical_rejected.count()
)

print("=" * 60)
print("OUTPUT DATAFRAMES")
print("df_ecommerce_valid")
print("df_physical_valid")
print("df_ecommerce_rejected")
print("df_physical_rejected")
print("=" * 60)

print("NEXT STEP: PRODUCT CATEGORY MAPPING")


In [0]:

# ============================================================
# NOTEBOOK 03 - PRODUCT CATEGORY MAPPING
# Project: Category-Level Demand Forecasting
# Objective: Map products to their respective categories
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. LOAD PREPARED SALES AND PRODUCT CATALOGS
# ------------------------------------------------------------

# Sales outputs from Notebook 02
# The DataFrames must be available in this notebook session.

required_sales = [
    "df_ecommerce_valid",
    "df_physical_valid"
]

missing_sales = [
    name for name in required_sales
    if name not in globals()
]

if missing_sales:
    raise ValueError(
        f"Missing sales DataFrames: {missing_sales}. "
        "Make the Notebook 02 outputs available before running "
        "this notebook. Separate Databricks notebooks do not "
        "automatically share Python variables."
    )

# Resolve catalog DataFrames from the loaded dataframes dictionary
# or from variables already available in the notebook.

def get_catalog(candidates, description):

    for candidate in candidates:

        if candidate in globals():
            return globals()[candidate]

        if candidate in dataframes:
            return dataframes[candidate]

    raise ValueError(
        f"Catalog not found: {description}. "
        f"Expected one of: {candidates}. "
        "Load the corresponding Silver table before continuing."
    )


df_ecommerce_products = get_catalog(
    [
        "df_ecommerce_produtos_silver",
        "df_ecommerce_produtos",
        "ecommerce_produtos"
    ],
    "ecommerce_produtos"
)

df_ecommerce_categories = get_catalog(
    [
        "df_ecommerce_categorias_silver",
        "df_ecommerce_categorias",
        "ecommerce_categorias"
    ],
    "ecommerce_categorias"
)

df_physical_products = get_catalog(
    [
        "df_physical_produtos_pereciveis_silver",
        "df_physical_produtos_pereciveis",
        "df_produtos",
        "physical_produtos_pereciveis"
    ],
    "physical_produtos_pereciveis"
)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

def validate_columns(df, required_columns, dataset_name):

    missing = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{dataset_name}: missing columns {missing}. "
            f"Available columns: {df.columns}"
        )


validate_columns(
    df_ecommerce_valid,
    ["product_id", "quantity_sold", "sale_date"],
    "E-commerce sales"
)

validate_columns(
    df_physical_valid,
    ["product_id", "quantity_sold", "sale_date"],
    "Physical sales"
)

validate_columns(
    df_ecommerce_products,
    ["sku", "id_categoria"],
    "E-commerce product catalog"
)

validate_columns(
    df_ecommerce_categories,
    ["id_categoria", "nome_categoria"],
    "E-commerce category catalog"
)

validate_columns(
    df_physical_products,
    ["sku", "categoria_pai"],
    "Physical product catalog"
)


# ------------------------------------------------------------
# 3. STANDARDIZE PRODUCT IDENTIFIERS
# ------------------------------------------------------------

# Keep the original product identifiers.
# Use normalized keys only for matching.

def normalize_product_key(column_name):

    return F.lower(
        F.trim(F.col(column_name).cast("string"))
    )


# ------------------------------------------------------------
# 4. PREPARE E-COMMERCE CATEGORY CATALOG
# ------------------------------------------------------------

ecommerce_category_columns = df_ecommerce_categories.columns

if "nome_categoria_pai" in ecommerce_category_columns:

    parent_category = F.when(
        F.trim(F.col("nome_categoria_pai")) != "",
        F.trim(F.col("nome_categoria_pai"))
    )

    category_expression = F.coalesce(
        parent_category,
        F.trim(F.col("nome_categoria"))
    )

else:

    category_expression = F.trim(F.col("nome_categoria"))


df_ecommerce_category_reference = (
    df_ecommerce_categories
    .select(
        F.col("id_categoria"),
        F.trim(F.col("nome_categoria")).alias("category_name"),
        category_expression.alias("category")
    )
    .filter(
        F.col("id_categoria").isNotNull()
        & F.col("category").isNotNull()
        & (F.col("category") != "")
    )
    .dropDuplicates(["id_categoria", "category"])
)


# ------------------------------------------------------------
# 5. IDENTIFY DUPLICATE CATEGORY IDENTIFIERS
# ------------------------------------------------------------

df_ecommerce_category_ambiguous = (
    df_ecommerce_category_reference
    .groupBy("id_categoria")
    .agg(
        F.countDistinct("category").alias("category_count")
    )
    .filter(F.col("category_count") > 1)
)

df_ecommerce_category_valid = (
    df_ecommerce_category_reference
    .join(
        df_ecommerce_category_ambiguous.select("id_categoria"),
        "id_categoria",
        "left_anti"
    )
    .dropDuplicates(["id_categoria"])
)


# ------------------------------------------------------------
# 6. MAP E-COMMERCE PRODUCTS TO CATEGORIES
# ------------------------------------------------------------

df_ecommerce_product_reference = (
    df_ecommerce_products
    .select(
        F.col("sku").cast("string").alias("catalog_product_id"),
        F.col("id_categoria")
    )
    .filter(
        F.col("sku").isNotNull()
        & (F.trim(F.col("sku")) != "")
    )
    .withColumn(
        "_product_key",
        normalize_product_key("catalog_product_id")
    )
    .join(
        df_ecommerce_category_valid,
        "id_categoria",
        "left"
    )
    .select(
        "_product_key",
        "catalog_product_id",
        "id_categoria",
        "category_name",
        "category"
    )
)

# Identify SKUs with conflicting category assignments.

df_ecommerce_product_ambiguous = (
    df_ecommerce_product_reference
    .filter(F.col("category").isNotNull())
    .groupBy("_product_key")
    .agg(
        F.countDistinct("category").alias("category_count")
    )
    .filter(F.col("category_count") > 1)
)

df_ecommerce_product_valid = (
    df_ecommerce_product_reference
    .join(
        df_ecommerce_product_ambiguous.select("_product_key"),
        "_product_key",
        "left_anti"
    )
    .dropDuplicates(["_product_key"])
)


# ------------------------------------------------------------
# 7. ENRICH E-COMMERCE SALES
# ------------------------------------------------------------

df_ecommerce_category_mapped = (
    df_ecommerce_valid
    .withColumn(
        "_product_key",
        normalize_product_key("product_id")
    )
    .join(
        df_ecommerce_product_valid.select(
            "_product_key",
            "id_categoria",
            "category_name",
            "category"
        ),
        "_product_key",
        "left"
    )
    .drop("_product_key")
    .withColumn(
        "flag_category_mapped",
        F.col("category").isNotNull()
    )
)

df_ecommerce_unmapped = (
    df_ecommerce_category_mapped
    .filter(~F.col("flag_category_mapped"))
)


# ------------------------------------------------------------
# 8. PREPARE PHYSICAL CATEGORY CATALOG
# ------------------------------------------------------------

physical_catalog_columns = df_physical_products.columns

physical_select = [
    F.col("sku").cast("string").alias("catalog_product_id"),
    F.trim(F.col("categoria_pai")).alias("category")
]

if "subcategoria" in physical_catalog_columns:

    physical_select.append(
        F.trim(F.col("subcategoria")).alias("subcategory")
    )

else:

    physical_select.append(
        F.lit(None).cast("string").alias("subcategory")
    )

df_physical_product_reference = (
    df_physical_products
    .select(*physical_select)
    .filter(
        F.col("catalog_product_id").isNotNull()
        & (F.trim(F.col("catalog_product_id")) != "")
        & F.col("category").isNotNull()
        & (F.col("category") != "")
    )
    .withColumn(
        "_product_key",
        normalize_product_key("catalog_product_id")
    )
)


# ------------------------------------------------------------
# 9. IDENTIFY AMBIGUOUS PHYSICAL PRODUCTS
# ------------------------------------------------------------

df_physical_product_ambiguous = (
    df_physical_product_reference
    .groupBy("_product_key")
    .agg(
        F.countDistinct("category").alias("category_count")
    )
    .filter(F.col("category_count") > 1)
)

df_physical_product_valid = (
    df_physical_product_reference
    .join(
        df_physical_product_ambiguous.select("_product_key"),
        "_product_key",
        "left_anti"
    )
    .dropDuplicates(["_product_key"])
)


# ------------------------------------------------------------
# 10. MAP PHYSICAL PRODUCTS TO CATEGORIES
# ------------------------------------------------------------

df_physical_category_mapped = (
    df_physical_valid
    .withColumn(
        "_product_key",
        normalize_product_key("product_id")
    )
    .join(
        df_physical_product_valid.select(
            "_product_key",
            "category",
            "subcategory"
        ),
        "_product_key",
        "left"
    )
    .drop("_product_key")
    .withColumn(
        "flag_category_mapped",
        F.col("category").isNotNull()
    )
)

df_physical_unmapped = (
    df_physical_category_mapped
    .filter(~F.col("flag_category_mapped"))
)


# ------------------------------------------------------------
# 11. E-COMMERCE MAPPING SUMMARY
# ------------------------------------------------------------

df_ecommerce_mapping_summary = (
    df_ecommerce_category_mapped
    .agg(
        F.count("*").alias("total_sales_records"),

        F.sum(
            F.when(F.col("flag_category_mapped"), 1).otherwise(0)
        ).alias("mapped_records"),

        F.sum(
            F.when(~F.col("flag_category_mapped"), 1).otherwise(0)
        ).alias("unmapped_records"),

        F.countDistinct("product_id").alias("distinct_products"),

        F.countDistinct(
            F.when(
                F.col("flag_category_mapped"),
                F.col("product_id")
            )
        ).alias("mapped_products")
    )
    .withColumn(
        "sales_channel",
        F.lit("ecommerce")
    )
    .withColumn(
        "mapping_percentage",
        F.round(
            F.col("mapped_records") /
            F.when(F.col("total_sales_records") > 0,
                   F.col("total_sales_records")).otherwise(F.lit(None))
            * 100,
            2
        )
    )
)


# ------------------------------------------------------------
# 12. PHYSICAL MAPPING SUMMARY
# ------------------------------------------------------------

df_physical_mapping_summary = (
    df_physical_category_mapped
    .agg(
        F.count("*").alias("total_sales_records"),

        F.sum(
            F.when(F.col("flag_category_mapped"), 1).otherwise(0)
        ).alias("mapped_records"),

        F.sum(
            F.when(~F.col("flag_category_mapped"), 1).otherwise(0)
        ).alias("unmapped_records"),

        F.countDistinct("product_id").alias("distinct_products"),

        F.countDistinct(
            F.when(
                F.col("flag_category_mapped"),
                F.col("product_id")
            )
        ).alias("mapped_products")
    )
    .withColumn(
        "sales_channel",
        F.lit("physical")
    )
    .withColumn(
        "mapping_percentage",
        F.round(
            F.col("mapped_records") /
            F.when(F.col("total_sales_records") > 0,
                   F.col("total_sales_records")).otherwise(F.lit(None))
            * 100,
            2
        )
    )
)


# ------------------------------------------------------------
# 13. DISPLAY E-COMMERCE RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("E-COMMERCE CATEGORY MAPPING")
print("=" * 60)

print("Mapped sales")
display(df_ecommerce_category_mapped)

print("Unmapped sales")
display(df_ecommerce_unmapped)

print("Ambiguous product identifiers")
display(df_ecommerce_product_ambiguous)

print("Ambiguous category identifiers")
display(df_ecommerce_category_ambiguous)

print("E-commerce mapping summary")
display(df_ecommerce_mapping_summary)


# ------------------------------------------------------------
# 14. DISPLAY PHYSICAL RESULTS
# ------------------------------------------------------------

print("=" * 60)
print("PHYSICAL CATEGORY MAPPING")
print("=" * 60)

print("Mapped sales")
display(df_physical_category_mapped)

print("Unmapped sales")
display(df_physical_unmapped)

print("Ambiguous product identifiers")
display(df_physical_product_ambiguous)

print("Physical mapping summary")
display(df_physical_mapping_summary)


# ------------------------------------------------------------
# 15. FINAL VALIDATION
# ------------------------------------------------------------

print("=" * 60)
print("PRODUCT CATEGORY MAPPING COMPLETED")
print("=" * 60)

print(
    "E-commerce mapped records:",
    df_ecommerce_category_mapped
    .filter(F.col("flag_category_mapped"))
    .count()
)

print(
    "E-commerce unmapped records:",
    df_ecommerce_unmapped.count()
)

print(
    "Physical mapped records:",
    df_physical_category_mapped
    .filter(F.col("flag_category_mapped"))
    .count()
)

print(
    "Physical unmapped records:",
    df_physical_unmapped.count()
)

print("=" * 60)
print("OUTPUT DATAFRAMES")
print("df_ecommerce_category_mapped")
print("df_ecommerce_unmapped")
print("df_physical_category_mapped")
print("df_physical_unmapped")
print("=" * 60)

print("NEXT STEP: MONTHLY DEMAND AGGREGATION")


In [0]:

# ============================================================
# NOTEBOOK 04 - MONTHLY CATEGORY DEMAND
# Project: Category-Level Demand Forecasting
# Objective: Aggregate sales into monthly category demand
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. VALIDATE INPUT DATAFRAMES
# ------------------------------------------------------------

required_dataframes = [
    "df_ecommerce_category_mapped",
    "df_physical_category_mapped"
]

missing_dataframes = [
    name for name in required_dataframes
    if name not in globals()
]

if missing_dataframes:
    raise ValueError(
        f"Missing DataFrames from Notebook 03: {missing_dataframes}"
    )


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "sale_date",
    "quantity_sold",
    "category"
]

def validate_columns(df, required, dataset_name):

    missing = [
        column for column in required
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{dataset_name}: missing columns {missing}. "
            f"Available columns: {df.columns}"
        )


validate_columns(
    df_ecommerce_category_mapped,
    required_columns,
    "E-commerce"
)

validate_columns(
    df_physical_category_mapped,
    required_columns,
    "Physical"
)


# ------------------------------------------------------------
# 3. STANDARDIZE SALES DATA
# ------------------------------------------------------------

def prepare_sales_dataframe(df, channel):

    return (
        df
        .select(
            F.to_date("sale_date").alias("sale_date"),
            F.col("quantity_sold").cast("double").alias(
                "quantity_sold"
            ),
            F.trim(F.col("category")).alias("category")
        )
        .withColumn("sales_channel", F.lit(channel))
    )


df_ecommerce_sales = prepare_sales_dataframe(
    df_ecommerce_category_mapped,
    "ecommerce"
)

df_physical_sales = prepare_sales_dataframe(
    df_physical_category_mapped,
    "physical"
)


# ------------------------------------------------------------
# 4. CHECK INVALID RECORDS
# ------------------------------------------------------------

def check_invalid_sales(df, dataset_name):

    invalid = df.filter(
        F.col("sale_date").isNull()
        | F.col("category").isNull()
        | (F.col("category") == "")
        | F.col("quantity_sold").isNull()
        | (F.col("quantity_sold") <= 0)
    )

    print(f"{dataset_name} - Invalid records:", invalid.count())

    display(invalid.limit(20))

    return invalid


df_ecommerce_invalid = check_invalid_sales(
    df_ecommerce_sales,
    "E-commerce"
)

df_physical_invalid = check_invalid_sales(
    df_physical_sales,
    "Physical"
)


# ------------------------------------------------------------
# 5. KEEP VALID SALES
# ------------------------------------------------------------

df_ecommerce_clean = df_ecommerce_sales.filter(
    F.col("sale_date").isNotNull()
    & F.col("category").isNotNull()
    & (F.col("category") != "")
    & F.col("quantity_sold").isNotNull()
    & (F.col("quantity_sold") > 0)
)

df_physical_clean = df_physical_sales.filter(
    F.col("sale_date").isNotNull()
    & F.col("category").isNotNull()
    & (F.col("category") != "")
    & F.col("quantity_sold").isNotNull()
    & (F.col("quantity_sold") > 0)
)


# ------------------------------------------------------------
# 6. COMBINE SALES CHANNELS
# ------------------------------------------------------------

df_sales_combined = (
    df_ecommerce_clean
    .unionByName(df_physical_clean)
    .withColumn(
        "month_start",
        F.trunc(F.col("sale_date"), "month")
    )
    .withColumn("year", F.year("month_start"))
    .withColumn("month", F.month("month_start"))
)

print("Combined sales records:", df_sales_combined.count())

display(df_sales_combined.limit(20))


# ------------------------------------------------------------
# 7. AGGREGATE MONTHLY DEMAND BY CATEGORY AND CHANNEL
# ------------------------------------------------------------

df_monthly_category_demand_by_channel = (
    df_sales_combined
    .groupBy(
        "month_start",
        "year",
        "month",
        "category",
        "sales_channel"
    )
    .agg(
        F.sum("quantity_sold").alias("monthly_demand"),
        F.count("*").alias("sales_records")
    )
    .orderBy(
        "month_start",
        "category",
        "sales_channel"
    )
)

print("Monthly demand by category and channel")

display(df_monthly_category_demand_by_channel)


# ------------------------------------------------------------
# 8. AGGREGATE TOTAL MONTHLY DEMAND BY CATEGORY
# ------------------------------------------------------------

df_monthly_category_demand = (
    df_monthly_category_demand_by_channel
    .groupBy(
        "month_start",
        "year",
        "month",
        "category"
    )
    .agg(
        F.sum("monthly_demand").alias("monthly_demand"),
        F.sum("sales_records").alias("sales_records")
    )
    .orderBy(
        "month_start",
        "category"
    )
)

print("Total monthly demand by category")

display(df_monthly_category_demand)


# ------------------------------------------------------------
# 9. CREATE A CONTINUOUS MONTHLY SERIES
# ------------------------------------------------------------
# Assumption:
# Months between the first and last observed sale for a category
# without records are treated as zero demand.
#
# This does not create months before the category's first
# observed sale.

df_category_periods = (
    df_monthly_category_demand
    .groupBy("category")
    .agg(
        F.min("month_start").alias("first_month"),
        F.max("month_start").alias("last_month")
    )
)

df_category_month_grid = (
    df_category_periods
    .withColumn(
        "month_start",
        F.explode(
            F.sequence(
                F.col("first_month"),
                F.col("last_month"),
                F.expr("INTERVAL 1 MONTH")
            )
        )
    )
    .select("category", "month_start")
)

df_monthly_category_demand_complete = (
    df_category_month_grid
    .join(
        df_monthly_category_demand,
        ["category", "month_start"],
        "left"
    )
    .withColumn(
        "monthly_demand",
        F.coalesce(F.col("monthly_demand"), F.lit(0.0))
    )
    .withColumn(
        "sales_records",
        F.coalesce(F.col("sales_records"), F.lit(0))
    )
    .withColumn("year", F.year("month_start"))
    .withColumn("month", F.month("month_start"))
    .orderBy("category", "month_start")
)

print("Continuous monthly demand series")

display(df_monthly_category_demand_complete)


# ------------------------------------------------------------
# 10. CHECK MONTHLY DATA COVERAGE
# ------------------------------------------------------------

df_monthly_coverage = (
    df_monthly_category_demand_complete
    .groupBy("category")
    .agg(
        F.count("*").alias("months_in_series"),
        F.min("month_start").alias("first_month"),
        F.max("month_start").alias("last_month"),
        F.sum(
            F.when(F.col("monthly_demand") == 0, 1).otherwise(0)
        ).alias("months_without_sales"),
        F.sum("monthly_demand").alias("total_demand")
    )
    .orderBy("category")
)

print("Monthly coverage by category")

display(df_monthly_coverage)


# ------------------------------------------------------------
# 11. CHECK DEMAND DISTRIBUTION
# ------------------------------------------------------------

df_demand_statistics = (
    df_monthly_category_demand_complete
    .groupBy("category")
    .agg(
        F.round(F.avg("monthly_demand"), 2).alias("avg_monthly_demand"),
        F.round(F.stddev("monthly_demand"), 2).alias("std_monthly_demand"),
        F.min("monthly_demand").alias("min_monthly_demand"),
        F.max("monthly_demand").alias("max_monthly_demand"),
        F.sum("monthly_demand").alias("total_demand")
    )
    .orderBy(F.desc("total_demand"))
)

print("Demand statistics by category")

display(df_demand_statistics)


# ------------------------------------------------------------
# 12. CHECK MONTHLY TOTALS
# ------------------------------------------------------------

df_monthly_total = (
    df_monthly_category_demand_complete
    .groupBy("month_start", "year", "month")
    .agg(
        F.sum("monthly_demand").alias("total_monthly_demand"),
        F.countDistinct("category").alias("active_categories")
    )
    .orderBy("month_start")
)

print("Total demand per month")

display(df_monthly_total)


# ------------------------------------------------------------
# 13. FINAL VALIDATION
# ------------------------------------------------------------

print("=" * 60)
print("NOTEBOOK 04 - FINAL VALIDATION")
print("=" * 60)

print(
    "E-commerce valid records:",
    df_ecommerce_clean.count()
)

print(
    "Physical valid records:",
    df_physical_clean.count()
)

print(
    "Combined sales records:",
    df_sales_combined.count()
)

print(
    "Monthly category records:",
    df_monthly_category_demand.count()
)

print(
    "Complete monthly records:",
    df_monthly_category_demand_complete.count()
)

print(
    "Number of categories:",
    df_monthly_category_demand_complete
    .select("category")
    .distinct()
    .count()
)

print(
    "First month:",
    df_monthly_category_demand_complete
    .agg(F.min("month_start"))
    .first()[0]
)

print(
    "Last month:",
    df_monthly_category_demand_complete
    .agg(F.max("month_start"))
    .first()[0]
)

print("=" * 60)
print("NOTEBOOK 04 COMPLETED")
print("=" * 60)

print("OUTPUT DATAFRAMES:")
print("df_monthly_category_demand_by_channel")
print("df_monthly_category_demand")
print("df_monthly_category_demand_complete")
print("df_monthly_coverage")
print("df_demand_statistics")
print("df_monthly_total")


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. VALIDATE INPUT
# ============================================================

if "df_monthly_category_demand_complete" not in globals():
    raise ValueError(
        "Input DataFrame not found. "
        "Run Notebook 04 or load its output first."
    )

df = df_monthly_category_demand_complete

required_columns = [
    "category",
    "month",
    "monthly_demand"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {df.columns}"
    )

# ============================================================
# 2. STANDARDIZE DATA TYPES
# ============================================================

df = (
    df
    .withColumn("month", F.to_date("month"))
    .withColumn(
        "monthly_demand",
        F.col("monthly_demand").cast("double")
    )
    .filter(
        F.col("category").isNotNull() &
        F.col("month").isNotNull() &
        F.col("monthly_demand").isNotNull()
    )
)

# One record per category and month is required.
duplicates = (
    df.groupBy("category", "month")
    .count()
    .filter(F.col("count") > 1)
)

if duplicates.limit(1).count() > 0:
    raise ValueError(
        "Duplicate category/month combinations found. "
        "Resolve them before feature engineering."
    )

# ============================================================
# 3. CALENDAR FEATURES
# ============================================================

df = (
    df
    .withColumn("year", F.year("month"))
    .withColumn("month_number", F.month("month"))
    .withColumn("quarter", F.quarter("month"))
    .withColumn(
        "time_index",
        F.year("month") * 12 + F.month("month")
    )
)

# ============================================================
# 4. DEFINE TEMPORAL WINDOW
# ============================================================

window_spec = (
    Window
    .partitionBy("category")
    .orderBy("month")
)

# ============================================================
# 5. LAG FEATURES
# ============================================================

df = (
    df
    .withColumn(
        "demand_lag_1",
        F.lag("monthly_demand", 1).over(window_spec)
    )
    .withColumn(
        "demand_lag_2",
        F.lag("monthly_demand", 2).over(window_spec)
    )
    .withColumn(
        "demand_lag_3",
        F.lag("monthly_demand", 3).over(window_spec)
    )
)

# ============================================================
# 6. ROLLING MEAN FEATURES
# ============================================================

# Excludes the current month to avoid data leakage.

window_3m = (
    window_spec
    .rowsBetween(-3, -1)
)

window_6m = (
    window_spec
    .rowsBetween(-6, -1)
)

df = (
    df
    .withColumn(
        "rolling_mean_3m",
        F.avg("monthly_demand").over(window_3m)
    )
    .withColumn(
        "rolling_mean_6m",
        F.avg("monthly_demand").over(window_6m)
    )
)

# ============================================================
# 7. MONTH-OVER-MONTH VARIATION
# ============================================================

df = (
    df
    .withColumn(
        "demand_change",
        F.col("monthly_demand") - F.col("demand_lag_1")
    )
    .withColumn(
        "demand_change_pct",
        F.when(
            F.col("demand_lag_1") > 0,
            (
                (F.col("monthly_demand") -
                 F.col("demand_lag_1")) /
                F.col("demand_lag_1")
            ) * 100
        )
    )
)

# ============================================================
# 8. HISTORY AVAILABILITY FLAGS
# ============================================================

df = (
    df
    .withColumn(
        "has_lag_1",
        F.col("demand_lag_1").isNotNull()
    )
    .withColumn(
        "has_lag_3",
        F.col("demand_lag_3").isNotNull()
    )
    .withColumn(
        "has_3m_history",
        F.col("rolling_mean_3m").isNotNull()
    )
    .withColumn(
        "has_6m_history",
        F.col("rolling_mean_6m").isNotNull()
    )
)

# ============================================================
# 9. FINAL DATAFRAME
# ============================================================

df_category_demand_features = df.orderBy(
    "category",
    "month"
)

# ============================================================
# 10. VALIDATION
# ============================================================

print("Feature engineering completed.")

print(
    "Total records:",
    df_category_demand_features.count()
)

print(
    "Total categories:",
    df_category_demand_features
    .select("category")
    .distinct()
    .count()
)

print("Available columns:")
print(df_category_demand_features.columns)

display(df_category_demand_features)

# ============================================================
# 11. FEATURE AVAILABILITY ANALYSIS
# ============================================================

df_feature_availability = (
    df_category_demand_features
    .agg(
        F.count("*").alias("total_records"),
        F.sum(
            F.col("has_lag_1").cast("int")
        ).alias("records_with_lag_1"),
        F.sum(
            F.col("has_lag_3").cast("int")
        ).alias("records_with_lag_3"),
        F.sum(
            F.col("has_3m_history").cast("int")
        ).alias("records_with_3m_history"),
        F.sum(
            F.col("has_6m_history").cast("int")
        ).alias("records_with_6m_history")
    )
)

display(df_feature_availability)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. VALIDATE INPUT
# ============================================================

if "df_category_demand_features" not in globals():
    raise ValueError(
        "DataFrame df_category_demand_features not found. "
        "Run Notebook 05 first."
    )

df = df_category_demand_features

required_columns = [
    "category",
    "month",
    "monthly_demand",
    "year",
    "month_number",
    "quarter",
    "time_index",
    "demand_lag_1",
    "demand_lag_2",
    "demand_lag_3",
    "rolling_mean_3m",
    "rolling_mean_6m"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

# ============================================================
# 2. DEFINE FEATURES AND TARGET
# ============================================================

target_column = "monthly_demand"

feature_columns = [
    "year",
    "month_number",
    "quarter",
    "time_index",
    "demand_lag_1",
    "demand_lag_2",
    "demand_lag_3",
    "rolling_mean_3m",
    "rolling_mean_6m"
]

# Important:
# monthly_demand is the target.
# demand_change and demand_change_pct are excluded
# because they use the current month's target.

# ============================================================
# 3. CHECK MONTHLY CONTINUITY
# ============================================================

window_spec = (
    Window
    .partitionBy("category")
    .orderBy("month")
)

df_check = (
    df
    .withColumn(
        "previous_month",
        F.lag("month").over(window_spec)
    )
    .withColumn(
        "expected_previous_month",
        F.add_months("month", -1)
    )
)

df_monthly_gaps = (
    df_check
    .filter(
        F.col("previous_month").isNotNull() &
        (
            F.col("previous_month") !=
            F.col("expected_previous_month")
        )
    )
    .select(
        "category",
        "month",
        "previous_month"
    )
)

if df_monthly_gaps.limit(1).count() > 0:
    print(
        "WARNING: Monthly gaps detected. "
        "Review the data before training."
    )
    display(df_monthly_gaps)
else:
    print("Monthly continuity check passed.")

# ============================================================
# 4. SELECT MODELING COLUMNS
# ============================================================

df_modeling_raw = df.select(
    "category",
    "month",
    *feature_columns,
    target_column
)

# ============================================================
# 5. FILTER RECORDS WITHOUT MINIMUM HISTORY
# ============================================================

# At least one previous month is required
# to generate a historical demand-based prediction.

df_modeling = (
    df_modeling_raw
    .filter(
        F.col("demand_lag_1").isNotNull()
    )
)

# ============================================================
# 6. VALIDATE TARGET AND FEATURES
# ============================================================

df_modeling = df_modeling.filter(
    F.col(target_column).isNotNull() &
    (F.col(target_column) >= 0)
)

# ============================================================
# 7. FINAL ORGANIZATION
# ============================================================

df_modeling = df_modeling.select(
    "category",
    "month",
    *feature_columns,
    target_column
).orderBy(
    "month",
    "category"
)

# ============================================================
# 8. DATASET SUMMARY
# ============================================================

total_records = df_modeling.count()

total_categories = (
    df_modeling
    .select("category")
    .distinct()
    .count()
)

print("Modeling dataset preparation completed.")
print(f"Total records: {total_records}")
print(f"Total categories: {total_categories}")
print(f"Target column: {target_column}")
print(f"Feature columns: {feature_columns}")

# ============================================================
# 9. CHECK NULL VALUES IN FEATURES
# ============================================================

null_expressions = [
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in feature_columns
]

df_feature_nulls = df_modeling.agg(*null_expressions)

display(df_feature_nulls)

# ============================================================
# 10. CHECK TARGET DISTRIBUTION
# ============================================================

display(
    df_modeling
    .select(target_column)
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    )
)

# ============================================================
# 11. FINAL DATASET
# ============================================================

display(df_modeling)
